In [36]:
import pandas as pd
import json
import glob

dataset_names = ["FullBody", "PartialBody", "FullAndPartialBodyMix"]

metadata_json_files = {}

for name in dataset_names:
    metadata_json_files[name] = glob.glob(f'../{name}Dataset/Metadata/*.json')

all_data = []

print(f"Found {sum(len(files) for files in metadata_json_files.values())} files. Processing...")

for dataset_name in metadata_json_files:
    for file_path in metadata_json_files[dataset_name]:
        with open(file_path, 'r') as f:
            data = json.load(f)
            
            # Flatten the nested 'Lesions' while keeping image-level metadata
            # meta=[] defines which root keys to repeat for every lesion row
            df_temp = pd.json_normalize(
                data, 
                record_path=['Lesions'], 
                meta=['ModelName', 'Filename', 'CameraAngle', 'Height', 'Width', 'NumberOfLesions'],
                errors='ignore'
            )

            df_temp['Dataset'] = dataset_name
            all_data.append(df_temp)

# Combine all dataframes into one
df = pd.concat(all_data, ignore_index=True)



Found 1398 files. Processing...


In [37]:
# Optional: Save to CSV for exploration in Excel
# df.to_csv("consolidated_lesion_data.csv", index=False)

### Total Lesions & Number of Unique Patients

In [38]:
print("\n--- Dataset Summary ---")
print(f"Total Lesions found: {len(df)}")
print(f"Unique Patients: {df['ModelName'].nunique()}")
print(df.head())



--- Dataset Summary ---
Total Lesions found: 33904
Unique Patients: 40
  Coordinate    LesionName  AnatomicalSite Diagnosis    ModelName  \
0    598,401  ISIC_0010858            Head     Nevus  WFemaleW000   
1   413,1431  ISIC_0000176  LowerExtremity  Melanoma  WFemaleW000   
2   597,1085  ISIC_0011105           Torso     Nevus  WFemaleW000   
3    566,649  ISIC_0012653           Torso     Nevus  WFemaleW000   
4    54,1021  ISIC_0000186  UpperExtremity     Nevus  WFemaleW000   

          Filename CameraAngle Height Width NumberOfLesions   Dataset  
0  478_WFemale.png       Right   2048  1024              46  FullBody  
1  478_WFemale.png       Right   2048  1024              46  FullBody  
2  478_WFemale.png       Right   2048  1024              46  FullBody  
3  478_WFemale.png       Right   2048  1024              46  FullBody  
4  478_WFemale.png       Right   2048  1024              46  FullBody  


### Patient Images

In [39]:
# patient ID
target_patient = 'WFemaleW000'

# Filter the dataframe for that patient
patient_table = df[df['ModelName'] == target_patient]
print(f"Diagnosis Summary:\n {patient_table['Diagnosis'].value_counts()}")

print(f"--- Images of Patient: {target_patient} ---")
patient_images = df[df['ModelName'] == target_patient][[
     'Filename', 'CameraAngle', 'NumberOfLesions', 'Dataset'
]].drop_duplicates(subset=['Filename'])
print(f"Number of Images: {len(patient_images)}")
print(f"Dataset Distribution\n {patient_images['Dataset'].value_counts()}")

print(patient_images) 

# TODO: how many unique lesions?

Diagnosis Summary:
 Diagnosis
Nevus                  1230
Melanoma                415
SeborrheicKeratosis     111
Name: count, dtype: int64
--- Images of Patient: WFemaleW000 ---
Number of Images: 60
Dataset Distribution
 Dataset
FullBody                 29
PartialBody              19
FullAndPartialBodyMix    12
Name: count, dtype: int64
              Filename CameraAngle NumberOfLesions                Dataset
0      478_WFemale.png       Right              46               FullBody
527    066_WFemale.png        Back              31               FullBody
938    495_WFemale.png        Left              42               FullBody
1419   378_WFemale.png       Front              28               FullBody
1560   167_WFemale.png        Left              58               FullBody
1868   057_WFemale.png        Back              62               FullBody
2586   292_WFemale.png        Back              24               FullBody
2665   072_WFemale.png       Front              60               Ful

### Patients Appear in Mutliple Datasets

In [42]:
# Identify patients found in more than 1 dataset
patient_overlap = df.groupby('ModelName')['Dataset'].nunique()
multi_dataset_patients = patient_overlap[patient_overlap > 1]

print(f"Number of patients appearing in multiple datasets: {len(multi_dataset_patients)}")
print(multi_dataset_patients)

# Create a pivot table to compare lesion counts across datasets
consistency_check = df.groupby(['ModelName', 'Dataset']).size().unstack()

# Filter to show only the overlapping patients found in step 1
overlapping_stats = consistency_check.loc[multi_dataset_patients.index]
print("\n--- Lesion Count Consistency ---")
print(overlapping_stats)

Number of patients appearing in multiple datasets: 26
ModelName
BFemaleW010    2
BFemaleW060    2
BFemaleW090    3
BMaleW090      2
WFemaleW000    3
Name: Dataset, dtype: int64

--- Lesion Count Consistency ---
Dataset      FullAndPartialBodyMix  FullBody  PartialBody
ModelName                                                
BFemaleW010                  125.0      89.0          NaN
BFemaleW060                  103.0     102.0          NaN
BFemaleW090                   44.0      68.0         56.0
BMaleW090                      NaN      54.0         25.0
WFemaleW000                  271.0    1215.0        270.0
WFemaleW010                  749.0     957.0         66.0
WFemaleW020                  453.0    1308.0        353.0
WFemaleW030                  468.0     896.0        256.0
WFemaleW040                  212.0     880.0        118.0
WFemaleW050                  525.0     862.0        167.0
WFemaleW060                  586.0     923.0        272.0
WFemaleW070                  678.0 

### Lesion Duplication

In [52]:
# 1. Count unique patients for every lesion name
lesion_sharing = df.groupby('LesionName')['ModelName'].nunique()

# 2. Filter for LesionNames associated with MORE than one patient
shared_lesions = lesion_sharing[lesion_sharing > 1].sort_values(ascending=False)

print(f"Found {len(shared_lesions)} 'colliding' lesion IDs.")
print("\n--- IDs shared by the most patients ---")
print(shared_lesions.head(10)) # Shows the ID and how many patients have it

# 3. Inspect a specific example of a collision
example_id = shared_lesions.index[0]
print(f"\n--- Detailed View: Patients sharing ID {example_id} ---")
lesion_table = df[df['LesionName'] == example_id][['ModelName', 'Dataset', 'AnatomicalSite', 'Diagnosis']]
print(f"Diagnosis Summary:\n {lesion_table['Diagnosis'].value_counts()}")
print(f"\n {lesion_table['AnatomicalSite'].value_counts()}")

print(lesion_table)


Found 1739 'colliding' lesion IDs.

--- IDs shared by the most patients ---
LesionName
ISIC_0014473    21
ISIC_0000255    21
ISIC_0012699    20
ISIC_0013713    20
ISIC_0015007    20
ISIC_0014696    20
ISIC_0013867    20
ISIC_0000337    19
ISIC_0011082    19
ISIC_0014931    19
Name: ModelName, dtype: int64

--- Detailed View: Patients sharing ID ISIC_0014473 ---
Diagnosis Summary:
 Diagnosis
Nevus    33
Name: count, dtype: int64

 AnatomicalSite
Torso    33
Name: count, dtype: int64
         ModelName                Dataset AnatomicalSite Diagnosis
1560   WFemaleW000               FullBody          Torso     Nevus
4372     WMaleW050               FullBody          Torso     Nevus
5143     WMaleW100               FullBody          Torso     Nevus
6742     WMaleW020               FullBody          Torso     Nevus
8351   WFemaleW070               FullBody          Torso     Nevus
8843   WFemaleW080               FullBody          Torso     Nevus
9236     WMaleW000               FullBody   